In [ ]:
using Plots
using LinearAlgebra
using ForwardDiff
using Printf
plotly()

f_base(v) = v[1]^2 + v[2]^2

const CONSTRAINTS = [
    v -> 2.0 - v[1],   # x ≥ 2.0
    v -> 2.0 - v[2],   # y ≥ 2.0
]

const R0       = [1.0, 1.0]
const PLOT_LIM = (-2.0, 7.0)

function Q_external(v, r_vec)
    penalty = sum(r_vec[i] * max(0.0, CONSTRAINTS[i](v))^2
                  for i in eachindex(CONSTRAINTS))
    return f_base(v) + penalty
end

function Q_along_ray(t, start, direction, r_vec)
    v = start .+ t .* direction
    return Q_external(v, r_vec)
end

function golden_section(φ, a, b; tol=1e-10, maxiter=1000)
    φr = (sqrt(5.0) + 1.0) / 2.0
    c  = b - (b - a) / φr
    d  = a + (b - a) / φr
    for _ in 1:maxiter
        abs(b - a) < tol && break
        φ(c) < φ(d) ? (b = d) : (a = c)
        c = b - (b - a) / φr
        d = a + (b - a) / φr
    end
    return (a + b) / 2.0
end

function find_zero_1d(g, a, b; tol=1e-8)
    for _ in 1:100
        m = (a + b) / 2
        g(a) * g(m) < 0 ? (b = m) : (a = m)
        abs(b - a) < tol && break
    end
    return (a + b) / 2
end

function compute_origin()
    v = [0.0, 0.0]
    for _ in 1:1000
        g = ForwardDiff.gradient(
            z -> sum(CONSTRAINTS[i](z)^2 for i in eachindex(CONSTRAINTS)), v)
        norm(g) < 1e-10 && break
        v -= 0.01 * g
    end
    return v
end

function compute_mult(v0; base_mult=2.0, scale=1.5)
    violations = [abs(CONSTRAINTS[i](v0)) for i in eachindex(CONSTRAINTS)]
    max_v = maximum(violations)
    max_v < 1e-10 && return fill(base_mult, length(CONSTRAINTS))
    return [base_mult + scale * (violations[i] / max_v)
            for i in eachindex(CONSTRAINTS)]
end

function compute_infeasible_start(origin, θ, dist)
    direction = [cos(θ), sin(θ)]
    v0 = origin .- dist .* direction

    any(CONSTRAINTS[i](v0) > 1e-6 for i in eachindex(CONSTRAINTS)) &&
        return v0, direction

    v0_alt = origin .+ dist .* direction
    if any(CONSTRAINTS[i](v0_alt) > 1e-6 for i in eachindex(CONSTRAINTS))
        return v0_alt, -direction
    end

    for sign in (-1.0, 1.0), ax in 1:2
        d = zeros(2)
        d[ax] = sign
        v_try = origin .+ dist .* d
        if any(CONSTRAINTS[i](v_try) > 1e-6 for i in eachindex(CONSTRAINTS))
            return v_try, -d
        end
    end

    return v0, direction
end

function fmt_r(r)
    abs(r) >= 1e6 ? @sprintf("%-10.2e", r) : @sprintf("%-10.4f", r)
end

function print_top(total_width, path_name, v0, mult)
    println()
    println("╔", "═"^total_width, "╗")
    println("║  ", rpad(path_name, total_width - 2), "║")
    println("╠", "═"^total_width, "╣")
    println("║  ", rpad(@sprintf("Старт:  x = %8.4f   y = %8.4f", v0[1], v0[2]),
            total_width - 2), "║")
    mult_parts = join(["r$i: $(round(mult[i], digits=4))"
                       for i in eachindex(mult)], "   ")
    println("║  ", rpad("Множители роста:  " * mult_parts, total_width - 2), "║")
    println("╠", "═"^total_width, "╣")
end

function print_header(total_width, r_vec)
    header = @sprintf("  %-5s", "iter")
    for i in eachindex(r_vec)
        header *= "  " * rpad("r$i", 10)
    end
    header *= @sprintf("  %-10s  %-10s  %-12s  %-12s",
                       "x*", "y*", "f(x*)", "штраф")
    println("║", rpad(header, total_width), "║")
    println("╠", "═"^total_width, "╣")
end

function print_row(total_width, iteration, r_vec, vmin, f_min, penalty)
    row = @sprintf("  %-5d", iteration)
    for i in eachindex(r_vec)
        row *= "  " * fmt_r(r_vec[i])
    end
    row *= @sprintf("  %-10.6f  %-10.6f  %-12.6f  %-12.8f",
                    vmin[1], vmin[2], f_min, penalty)
    println("║", rpad(row, total_width), "║")
end

function print_bottom(total_width, traj_x, traj_y)
    println("╠", "═"^total_width, "╣")
    result = @sprintf("  Минимум:  x* = %.8f   y* = %.8f   f(x*) = %.8f",
                      traj_x[end], traj_y[end],
                      f_base([traj_x[end], traj_y[end]]))
    println("║", rpad(result, total_width), "║")
    println("╚", "═"^total_width, "╝")
end

function plot_for_path(θ, path_name)
    r_vec  = copy(R0)
    tol    = 1e-6
    origin = compute_origin()

    dist = 3.0
    v0, direction = compute_infeasible_start(origin, θ, dist)
    MULT = compute_mult(v0)

    lo, hi = PLOT_LIM
    xs = range(lo, hi, length=100)
    ys = range(lo, hi, length=100)

    colors_list = [:red, :blue, :green, :orange, :purple,
                   :brown, :magenta, :cyan, :gold]
    cmaps_list  = [:viridis, :plasma, :inferno, :magma,
                   :blues, :greens, :thermal, :haline]

    p = plot(layout=(1, 2), size=(1400, 650))

    plot!(p[1],
        title  = path_name * " (3D)",
        xlabel = "x", xlims = (lo, hi),
        ylabel = "y", ylims = (lo, hi),
        zlabel = "Q(x,y,r)",
        camera = (45, 30),
        legend = :topright
    )
    plot!(p[2],
        title  = path_name * " (сверху)",
        xlabel = "x", xlims = (lo, hi),
        ylabel = "y", ylims = (lo, hi),
        legend = :topright
    )

    for (i, g) in enumerate(CONSTRAINTS)
        if i == 1
            bnd = find_zero_1d(t -> g([t, origin[2]]), lo, hi)
            plot!(p[2], [bnd, bnd], [lo, hi],
                  linecolor=:red, linewidth=2, linestyle=:dash,
                  label="g$i=0  (x=$(round(bnd, digits=2)))")
        elseif i == 2
            bnd = find_zero_1d(t -> g([origin[1], t]), lo, hi)
            plot!(p[2], [lo, hi], [bnd, bnd],
                  linecolor=:darkorange, linewidth=2, linestyle=:dash,
                  label="g$i=0  (y=$(round(bnd, digits=2)))")
        end
    end

    Z_base = [f_base([x, y]) for y in ys, x in xs]
    surface!(p[1], xs, ys, Z_base,
             alpha=0.15, cmap=:grays, colorbar=false, label=false)

    t_ray = range(-dist - 0.5, dist + 2, length=200)
    ray_x = origin[1] .+ t_ray .* direction[1]
    ray_y = origin[2] .+ t_ray .* direction[2]
    plot!(p[2], ray_x, ray_y,
          linecolor=:black, linewidth=2, linestyle=:dot,
          label="луч θ=$(round(θ*180/π, digits=0))°")

    scatter!(p[1], [v0[1]], [v0[2]], [f_base(v0)],
             markersize=10, markercolor=:limegreen, markershape=:diamond,
             markerstrokecolor=:black, label=false)
    scatter!(p[2], [v0[1]], [v0[2]],
             markersize=10, markercolor=:limegreen, markershape=:diamond,
             markerstrokecolor=:black, label="старт")

    n_r         = length(r_vec)
    total_width = 6 + n_r*12 + 12 + 12 + 14 + 14 + 4

    print_top(total_width, path_name, v0, MULT)
    print_header(total_width, r_vec)

    traj_x, traj_y, traj_z          = Float64[], Float64[], Float64[]
    all_pts_x, all_pts_y, all_pts_z = Float64[v0[1]], Float64[v0[2]], Float64[f_base(v0)]
    iter_colors = []

    color_idx = 1
    iteration = 1
    t_lo, t_hi = -0.5, dist + 3.0

    while iteration < 200
        current_color = colors_list[mod1(color_idx, length(colors_list))]
        current_cmap  = cmaps_list[mod1(color_idx, length(cmaps_list))]

        φ(t) = Q_along_ray(t, v0, direction, r_vec)

        t_min   = golden_section(φ, t_lo, t_hi)
        vmin    = v0 .+ t_min .* direction
        f_min   = f_base(vmin)
        penalty = sum(r_vec[i] * max(0.0, CONSTRAINTS[i](vmin))^2
                      for i in eachindex(CONSTRAINTS))

        t_parab = range(t_lo, t_hi, length=300)
        par_x   = v0[1] .+ t_parab .* direction[1]
        par_y   = v0[2] .+ t_parab .* direction[2]
        par_z   = [φ(t) for t in t_parab]

        z_clip = minimum(Z_base) + (maximum(Z_base) - minimum(Z_base)) * 3
        par_z  = clamp.(par_z, -Inf, z_clip)

        plot!(p[1], par_x, par_y, par_z,
              linewidth=2.5, color=current_color, label=false)

        Z_q = [Q_external([x, y], r_vec) for y in ys, x in xs]
        Z_q = clamp.(Z_q, -Inf, z_clip)
        surface!(p[1], xs, ys, Z_q,
                 alpha=0.2, cmap=current_cmap, colorbar=false, label=false)

        contour!(p[2], xs, ys, Z_q,
                 seriescolor=current_color, levels=8, linewidth=2,
                 label=false)

        plot!(p[2], [hi + 100, hi + 101], [hi + 100, hi + 101],
              linewidth=2.5, color=current_color,
              label="парабола r=$(round(r_vec[1], digits=2))")

        push!(traj_x, vmin[1])
        push!(traj_y, vmin[2])
        push!(traj_z, f_min)
        push!(all_pts_x, vmin[1])
        push!(all_pts_y, vmin[2])
        push!(all_pts_z, f_min)
        push!(iter_colors, current_color)

        print_row(total_width, iteration, r_vec, vmin, f_min, penalty)

        if length(traj_x) > 1 &&
           norm([traj_x[end]-traj_x[end-1], traj_y[end]-traj_y[end-1]]) < tol
            break
        end

        v0        = vmin
        t_lo      = -1.0
        t_hi      = dist + 1.0
        r_vec     = r_vec .* MULT
        color_idx += 1
        iteration += 1
    end

    print_bottom(total_width, traj_x, traj_y)

    if length(traj_x) > 1
        plot!(p[1], all_pts_x, all_pts_y, all_pts_z,
              linewidth=3, linestyle=:dash, color=:black, label=false)
        plot!(p[2], all_pts_x, all_pts_y,
              linewidth=3, linestyle=:dash, color=:black, label="траектория")

        scatter!(p[1], [all_pts_x[1]], [all_pts_y[1]], [all_pts_z[1]],
                 markersize=10, markercolor=:limegreen, markershape=:diamond,
                 markerstrokecolor=:black, label=false)
        scatter!(p[2], [all_pts_x[1]], [all_pts_y[1]],
                 markersize=10, markercolor=:limegreen, markershape=:diamond,
                 markerstrokecolor=:black, label=false)

        for k in 2:length(all_pts_x)-1
            c = iter_colors[k-1]
            scatter!(p[1], [all_pts_x[k]], [all_pts_y[k]], [all_pts_z[k]],
                     markersize=7, markercolor=c, markershape=:circle,
                     markerstrokecolor=:black, label=false)
            scatter!(p[2], [all_pts_x[k]], [all_pts_y[k]],
                     markersize=7, markercolor=c, markershape=:circle,
                     markerstrokecolor=:black, label=false)
        end

        scatter!(p[2], [all_pts_x[end]], [all_pts_y[end]],
                 markersize=14, markercolor=:red, markershape=:pentagon,
                 markerstrokecolor=:black, label="минимум")

        ex, ey, ez = all_pts_x[end], all_pts_y[end], all_pts_z[end]
        d = 0.25
        for (dx, dy) in [(d, 0.0), (-d, 0.0), (0.0, d), (0.0, -d),
                         (d*0.7, d*0.7), (-d*0.7, d*0.7),
                         (d*0.7, -d*0.7), (-d*0.7, -d*0.7)]
            plot!(p[1], [ex, ex+dx], [ey, ey+dy], [ez, ez],
                  linewidth=2.5, color=:yellow, label=false)
        end
        scatter!(p[1], [ex], [ey], [ez],
                 markersize=14, markercolor=:red,
                 markerstrokecolor=:yellow, markerstrokewidth=3,
                 label=false)
    end

    return p
end

for deg in 0:15:90
    θ = π + deg * π / 180
    name = if deg == 0
        "Траектория вдоль оси X"
    elseif deg == 90
        "Траектория вдоль оси Y"
    else
        "Траектория под $(deg)°"
    end
    p = plot_for_path(θ, name)
    display(p)
end